# 02 · Interaction Queue：100 条冲突如何收敛成 5 个问题

**目标读者**：贡献者、产品设计者

**核心目的**：证明问题队列的价值不在“发现脏数据”，而在**把上百条冲突压成人能答完的几道题**。本 notebook 用 150 行含 5 类脏点的数据，走完 `open → start → next → answer → export`，重点看三件事：聚合层级、批量套用、会话上限。

**验收标准**

1. 100+ 条脏点收敛为 **≤5 个问题**，层级分布恰为 `strategy 1 / column 3 / point 1`；
2. 一个 `batch_apply` 决策 = **1 条事件 + N 条单元格变更**，N ≥ 100，全部共享同一 `decision_id`；
3. 三个 column 级问题各自展开 ≥3 条变更；
4. 聚合后的 point 问题内**同时存在** `changed_value` 与 `flagged_kept`（同一决策、两种处置）；
5. 会话上限设为 2 时，剩余问题转入 `session.unanswered`，主会话队列不受影响；
6. `out/02/` 下 8 个文件齐全（7 产物 + `input_dirty.csv`），且 `engine.py` 函数体零金融词。

## 大纲

1. 造 150 行脏数据：120 行重复主键 + 3 行 High<Low + 5 行单位错位 + 4 行缺失 token + 10 行日期格式混杂
2. `start()`：检测 → 聚合 → 排序，验证队列收敛为 5 题
3. strategy 问题：一次 `batch_apply` 产出一条决策 + 100+ 条 `dropped_row`
4. 3 个 column 问题：列级批量，各自展开多条变更
5. 1 个聚合后的 point 问题：逐行指定处置，修正与打标并存
6. 会话上限：`max_questions_per_session=2` 触发 `unanswered`
7. 导出与汇总断言：8 个文件、护照字段、diff 四色、engine 零金融词

In [1]:
import ast, json
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

import finpass as fp

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
OUT02 = ROOT / 'out' / '02'
OUT02.mkdir(parents=True, exist_ok=True)
INPUT02 = OUT02 / 'input_dirty.csv'

ROWS = 150
idx = np.arange(ROWS)
dup_keys = pd.date_range('2024-01-02', periods=12, freq='B').strftime('%Y-%m-%d').tolist()
unique_keys = pd.date_range('2024-03-01', periods=30, freq='B').strftime('%Y-%m-%d').tolist()

df = pd.DataFrame({
    '日期': dup_keys * 10 + unique_keys,
    '开盘价': np.round(10.0 + (idx % 7) * 0.1, 2),
    '最高价': np.round(10.5 + (idx % 5) * 0.1, 2),
    '最低价': np.round(9.5 + (idx % 5) * 0.1, 2),
    '收盘价': np.round(10.2 + (idx % 6) * 0.1, 2),
    '成交量_股': 1_000_000 + idx * 1000,
    '成交额_元': 1400 + (idx % 40) * 10,
})
df['收盘价'] = df['收盘价'].astype(object)          # 允许后面写入 '--'
df.loc[100:102, '最高价'] = 9.00                    # 3 行 High < Low
df.loc[100:102, '最低价'] = 9.80
df.loc[110:114, '成交额_元'] = df.loc[110:114, '成交额_元'] * 10000   # 5 行单位错 10000 倍
df.loc[120:123, '收盘价'] = '--'                    # 4 行缺失 token
df.loc[130:134, '日期'] = pd.to_datetime(df.loc[130:134, '日期']).dt.strftime('%Y/%m/%d')
df.loc[135:139, '日期'] = pd.to_datetime(df.loc[135:139, '日期']).dt.strftime('%Y%m%d')
df.to_csv(INPUT02, index=False)

print('shape', df.shape)
print('重复主键行', int(df.duplicated('日期').sum()), '| High<Low 3 | 单位异常 5 | 缺失 token 4 | 非 ISO 日期 10')

shape (150, 7)
重复主键行 108 | High<Low 3 | 单位异常 5 | 缺失 token 4 | 非 ISO 日期 10


In [2]:
session = fp.open(INPUT02, max_questions_per_session=50)
summary = session.start()
display(summary)

counts = Counter(q.aggregation_level for q in session.queue)
for q in session.queue:
    print(f'  [{q.aggregation_level:8}] {q.rule_id:20} 影响 {len(q.affected_row_keys):3} 个主键   预估变更 {q.estimated_changes}')

assert len(session.queue) == 5, len(session.queue)
assert counts == Counter({'strategy': 1, 'column': 3, 'point': 1}), counts
assert summary['total_questions'] == 5
assert set(summary['rules_hit']) == {'missing_token', 'duplicate_key', 'ohlc_contradiction', 'unit_ambiguity', 'date_format'}

point_q = next(q for q in session.queue if q.aggregation_level == 'point')
assert len(point_q.affected_row_keys) == 3
assert len(point_q.context_samples) == 3
print('C2 通过：100+ 条脏点收敛成 5 个问题')

{'total_questions': 5,
 'strategy': 1,
 'column': 3,
 'point': 1,
 'total_affected_rows': 34,
 'rules_hit': ['date_format',
  'duplicate_key',
  'missing_token',
  'ohlc_contradiction',
  'unit_ambiguity'],
 'unanswered_count': 0}

  [strategy] duplicate_key        影响  12 个主键   预估变更 12
  [column  ] missing_token        影响   4 个主键   预估变更 4
  [column  ] unit_ambiguity       影响   5 个主键   预估变更 5
  [column  ] date_format          影响  10 个主键   预估变更 10
  [point   ] ohlc_contradiction   影响   3 个主键   预估变更 3
C2 通过：100+ 条脏点收敛成 5 个问题


In [3]:
q_dup = next(q for q in session.queue if q.rule_id == 'duplicate_key')
display(q_dup.render())
session.answer(q_dup.question_id, 'batch_apply', {'keep': '保留第一条'}, '重复主键共 108 行，批量保留第一条')

event = session.events[-1]
batch_changes = [c for c in session.changes if c['decision_id'] == event['event_id']]
print('一次 batch 决策 ->', len(batch_changes), '条变更，全部 dropped_row:',
      all(c['change_type'] == 'dropped_row' for c in batch_changes))

assert event['batch'] is True
assert len(batch_changes) >= 100
assert all(c['change_type'] == 'dropped_row' for c in batch_changes)
print('C3 通过：1 条事件 = 1 个 decision_id =', len(batch_changes), '条单元格变更')

'┌─ [strategy] duplicate_key  (q_dup_key)\n│ 影响行数：12    预估变更：12\n│ 样本：日期=2024-01-02\n│ 样本：日期=2024-01-03\n│ 样本：日期=2024-01-04\n│ 样本：日期=2024-01-05\n│ 样本：日期=2024-01-08\n│ 可选答案：保留第一条 / 保留最后一条 / 跳过打标\n│ 默认建议：保留第一条\n└──────────────────────────────────────────────────────────'

一次 batch 决策 -> 108 条变更，全部 dropped_row: True
C3 通过：1 条事件 = 1 个 decision_id = 108 条单元格变更


In [4]:
for q in [x for x in session.queue if x.aggregation_level == 'column']:
    display(q.render())
    if q.rule_id == 'missing_token':
        params = {'fill': 'NaN'}
    elif q.rule_id == 'unit_ambiguity':
        params = {'scale': 0.0001, 'unit': '万元'}
    else:
        params = {'fmt': '%Y-%m-%d'}
    session.answer(q.question_id, 'batch_apply', params, f'{q.rule_id} 列级批量处理')
    did = session.events[-1]['event_id']
    n = sum(1 for c in session.changes if c['decision_id'] == did)
    print(f'  {q.rule_id}: {n} 条变更')
    assert n >= 3, (q.rule_id, n)
print('C4 通过：3 个列级问题各自展开为多条变更')

'┌─ [column] missing_token  (q_missing_收盘价)\n│ 影响行数：4    预估变更：4\n│ 样本：日期=2024-03-01  收盘价=--\n│ 样本：日期=2024-03-04  收盘价=--\n│ 样本：日期=2024-03-05  收盘价=--\n│ 样本：日期=2024-03-06  收盘价=--\n│ 可选答案：填 NaN / 跳过打标\n│ 默认建议：填 NaN\n└──────────────────────────────────────────────────────────'

  missing_token: 4 条变更


'┌─ [column] unit_ambiguity  (q_unit_成交额_元)\n│ 影响行数：5    预估变更：5\n│ 样本：日期=2024-01-04  成交额_元=17000000\n│ 样本：日期=2024-01-05  成交额_元=17100000\n│ 样本：日期=2024-01-08  成交额_元=17200000\n│ 样本：日期=2024-01-09  成交额_元=17300000\n│ 样本：日期=2024-01-10  成交额_元=17400000\n│ 可选答案：按万元折算异常行 / 跳过打标\n│ 默认建议：按万元折算异常行\n└──────────────────────────────────────────────────────────'

  unit_ambiguity: 5 条变更


'┌─ [column] date_format  (q_date_日期)\n│ 影响行数：10    预估变更：10\n│ 样本：日期=2024/03/15\n│ 样本：日期=2024/03/18\n│ 样本：日期=2024/03/19\n│ 样本：日期=2024/03/20\n│ 样本：日期=2024/03/21\n│ 可选答案：转换为 %Y-%m-%d / 跳过打标\n│ 默认建议：转换为 %Y-%m-%d\n└──────────────────────────────────────────────────────────'

  date_format: 10 条变更
C4 通过：3 个列级问题各自展开为多条变更


In [5]:
q_point = next(q for q in session.queue if q.aggregation_level == 'point')
display(q_point.render())
print('聚合样本数', len(q_point.context_samples), '| 影响主键', q_point.affected_row_keys)

rows = dict(zip(q_point.affected_row_keys, ['answer', 'answer', 'skip']))
session.answer(q_point.question_id, 'batch_apply', {'rows': rows}, '前两行以收盘价修正，第三行跳过打标')

did = session.events[-1]['event_id']
point_changes = [c for c in session.changes if c['decision_id'] == did]
flagged = [c for c in point_changes if c['change_type'] == 'flagged_kept']
fixed = [c for c in point_changes if c['change_type'] == 'changed_value']
print('point 决策 ->', len(point_changes), '条变更：changed_value', len(fixed), '| flagged_kept', len(flagged))

assert len(flagged) > 0 and len(fixed) > 0
print('C5 通过：1 个聚合问题内并存修正与打标')

'┌─ [point] ohlc_contradiction  (q_ohlc_2024-01-08)\n│ 影响行数：3    预估变更：3\n│ 样本：日期=2024-01-08  最高价=9.0  最低价=9.8  收盘价=10.6\n│ 样本：日期=2024-01-09  最高价=9.0  最低价=9.8  收盘价=10.7\n│ 样本：日期=2024-01-10  最高价=9.0  最低价=9.8  收盘价=10.2\n│ 可选答案：标记跳过 / 以收盘价为准修正\n│ 默认建议：标记跳过\n└──────────────────────────────────────────────────────────'

聚合样本数 3 | 影响主键 ['2024-01-08', '2024-01-09', '2024-01-10']
point 决策 -> 8 条变更：changed_value 4 | flagged_kept 4
C5 通过：1 个聚合问题内并存修正与打标


In [6]:
session2 = fp.open(INPUT02, max_questions_per_session=2)
session2.start()

served = 0
while (q2 := session2.next()) is not None:
    session2.answer(q2.question_id, 'skip', {}, '演示：达到会话上限')
    served += 1

print('已答', served, '题 | unanswered', len(session2.unanswered))
for item in session2.unanswered:
    print('  ', item.question_id, item.rule_id, item.aggregation_level, item.reason)

assert served == 2
assert len(session2.unanswered) > 0
assert all(u.reason == 'max_questions_exceeded' for u in session2.unanswered)
assert len(session.queue) == 5
assert len(session2.df) == len(session.original)
print('C6 通过：上限把剩余问题转成 unanswered，不污染主会话')

已答 2 题 | unanswered 3
   q_unit_成交额_元 unit_ambiguity column max_questions_exceeded
   q_date_日期 date_format column max_questions_exceeded
   q_ohlc_2024-01-08 ohlc_contradiction point max_questions_exceeded
C6 通过：上限把剩余问题转成 unanswered，不污染主会话


In [7]:
session.export(OUT02)
files = sorted(p.name for p in OUT02.iterdir())
expected = sorted(['input_dirty.csv', 'input_dirty.clean.csv', 'input_dirty.passport.json',
                   'input_dirty.events.jsonl', 'input_dirty.changes.parquet',
                   'input_dirty.report.before.json', 'input_dirty.report.after.json',
                   'input_dirty.diff.html'])
print(len(files), files)
assert files == expected, files

passport = json.loads((OUT02 / 'input_dirty.passport.json').read_text(encoding='utf-8'))
for field in ['input_hash', 'output_file_sha256', 'human_decisions_count', 'unresolved_issues_count', 'auto_only']:
    assert field in passport, field

events = [json.loads(line) for line in (OUT02 / 'input_dirty.events.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]
assert len(events) == 5
assert sum(1 for e in events if e['batch']) == 5

changes = pd.read_parquet(OUT02 / 'input_dirty.changes.parquet')
assert len(changes) > 0
assert changes['decision_id'].notna().all()

html = (OUT02 / 'input_dirty.diff.html').read_text(encoding='utf-8')
for color in ['#d4f8d4', '#fff3cd', '#d1ecf1', '#f8d7da', '#ffe8b3']:
    assert color in html, color

engine_src = (ROOT / 'src' / 'finpass' / 'engine.py').read_text(encoding='utf-8')
body = engine_src.replace(ast.get_docstring(ast.parse(engine_src)) or '', '')
FIN_WORDS = ['收盘', '开盘', '涨停', '跌停', '复权', '成交量', '成交额', '最高价', '最低价']
hits = [w for w in FIN_WORDS if w in body]
assert not hits, hits

print(f'✅ 02 验收通过 | 5 题 / {len(events)} 决策 / {len(changes)} 变更 / {len(files)} 文件 / 五色齐全')

8 ['input_dirty.changes.parquet', 'input_dirty.clean.csv', 'input_dirty.csv', 'input_dirty.diff.html', 'input_dirty.events.jsonl', 'input_dirty.passport.json', 'input_dirty.report.after.json', 'input_dirty.report.before.json']
✅ 02 验收通过 | 5 题 / 5 决策 / 135 变更 / 8 文件 / 五色齐全
